# Interactive DFG: the steps on the running example

This notebook shows every step of the page "Interactive DFG" (`docs/pages/interactive-dfg.md`) on the running example (one slab, five coils, 17 events).
It only imports and calls the package functions.
Every step is a plain polars function on three frames of the log: events (`event`, `time`, `activity`), objects (`object`, `object_type`) and the event-to-object relations (`event`, `object`, `object_type`).

In [ ]:
from pathlib import Path

import polars as pl
from ocelescope import OCEL

from ocelescope_module_interactive_dfg.dfg.arcs import arc_count_filter, arc_share_filter, directly_follows_arcs
from ocelescope_module_interactive_dfg.dfg.connectivity import stay_connected
from ocelescope_module_interactive_dfg.dfg.graph import (
    TypeSetting,
    default_selection,
    directly_follows_graph,
    log_frames,
    type_overview,
)
from ocelescope_module_interactive_dfg.dfg.nodes import activity_frequencies, involved_object_types, shown_activities
from ocelescope_module_interactive_dfg.dfg.projection import default_activities, project_traces
from ocelescope_module_interactive_dfg.dfg.traces import considered_objects, object_traces
from ocelescope_module_interactive_dfg.dfg.variants import object_variants, variant_filter, variant_ranking

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(80)

## Input: the log's frames (`log_frames`)
`log_frames(ocel)` reads events, objects and e2o from Ocelescope's DuckDB tables (as polars frames) and counts the events per object type and activity (`type_activities`, used for the default activities).

In [ ]:
EXAMPLE = Path.cwd().parent / "data" / "running_example" / "running_example.sqlite"
frames = log_frames(OCEL.read(EXAMPLE))
frames.events

## Step 0: the selected object types (`type_overview`, `default_selection`)
| Parameter | Meaning |
|---|---|
| `objects` | all objects (`object`, `object_type`) |
| `n` | number of object types, default 5 |

The default selection is the `n` object types with the most objects (ties by name).
The running example has only two object types, so both are selected: coil (5 objects) and slab (1).

In [ ]:
display(type_overview(frames.objects))
[s.object_type for s in default_selection(frames.objects)]

## Step 1: traces (`considered_objects`, `object_traces`)
Every object of a selected type contributes its trace: its events ordered by time, then activity name, then event id (`seq`).
An object linked to an event with several qualifiers counts once.

In [ ]:
types = ["coil", "slab"]
traces = object_traces(frames.e2o, frames.events, considered_objects(frames.objects, types))
traces

## Step 2: activity projection (`default_activities`, `project_traces`)
| Parameter | Meaning |
|---|---|
| `counts` | events per object type and activity (`frames.type_activities`) |
| `object_types` | the types to preselect for |
| `shared_from` | types whose defaults are shared with all (not used on the page) |

Default per type: its most frequent 40 % of activities, rounded up, at least 3, at most 10.
coil has 7 activities, so 3 are preselected: crane move (4 events), quality check (4), load truck (2; before unload truck by first occurrence).
`project_traces` removes the events of all other activities from the traces of that type before anything else.

In [ ]:
included = default_activities(frames.type_activities, types)
display(included)
projected = project_traces(traces, included)
projected

## Step 3: variants (`object_variants`, `variant_ranking`, `variant_filter`)
A variant is the sequence of activities of a (projected) trace.
Variants are ranked by their number of objects, then by first occurrence, then alphabetically; a variant is kept while the variants ranked before it cover less than π % of the type's objects (default 80 %).

In [ ]:
ranking = variant_ranking(object_variants(projected))
display(ranking)
kept = variant_filter(projected, {"coil": 80, "slab": 80})
kept.select("object").unique().sort("object")

## Step 4: arcs (`directly_follows_arcs`, `arc_share_filter`, `arc_count_filter`)
Every included trace contributes start → first activity, each pair of consecutive activities and last activity → end (null = start or end node).
Arcs (projection) α keeps the first ⌈α/100 · n⌉ arcs of a type ranked by objects, then traversals, plus every arc tied with the last kept one (default 50 %).
Arcs (display) N keeps exactly N arcs (default: all).

In [ ]:
arcs = directly_follows_arcs(kept)
display(arcs)
threshold_arcs = arc_share_filter(arcs, {"coil": 50, "slab": 50})
display(threshold_arcs)
arc_count_filter(threshold_arcs, {}, total=3)

## Step 5: Stay connected (`stay_connected`)
Adds back the fewest, most frequent arcs so that every shown arc lies on a path from the start to the end node of its type (`connecting` = True).

In [ ]:
stay_connected(arcs, arc_count_filter(threshold_arcs, {}, total=3))

## Step 6: nodes (`shown_activities`, `activity_frequencies`, `involved_object_types`)
The frequency of an activity is the number of its events on the included traces; `involved_object_types` gives, per object type, the events and objects of these events (unselected types become circles on the node).

In [ ]:
shown = shown_activities(threshold_arcs)
display(activity_frequencies(kept, shown))
involved_object_types(kept, shown, frames.e2o)

## All steps in one call (`directly_follows_graph`)
With the defaults (the selection of step 0, the default activities, variants 80 %, Arcs (projection) 50 %, Stay connected on).
The page calls the same function with all arcs and applies steps 4 and 5 in the browser.

In [ ]:
graph = directly_follows_graph(frames)
pl.DataFrame([a.model_dump() for a in graph.arcs])